# Week 3-1 · MMT-01 — Execution Strategy I (Orders & Order Books)
**Practice notebook — real lecture numbers + a labelled simulated order book.**

This lecture is *market microstructure*: how an order actually reaches the exchange, how the
**order book** stores bids & asks, what the **order types** (market / limit / stop / iceberg / pegging /
discretion) do, and how execution desks measure quality with **VWAP** and a **time-horizon** rule.

The lecture's shipped data is a tiny Excel VWAP table (4 trades) and three order-book reading
exercises. There is **no tick-by-tick CSV** (real order books are not in the course pack and
`yfinance` does not expose depth), so wherever we need a live book we build a **clearly-labelled
simulated order book** and reproduce the exact slide/Excel numbers everywhere they exist.

**What we reproduce (all numbers come straight from the lecture):**
1. The order book — best bid/ask, spread, depth (slide values $49.90 / $50.10 …)
2. A **market order walking the book** → average fill price (step-by-step table)
3. **VWAP** = 14.205 (the exact 4-row Excel table) + moving VWAP
4. Execution **time-horizon** rule: TTH = 10000/(0.15×500) = 134 min → PTH = 120
5. **Order-flow-imbalance (OFI)** → linear price impact (Cont–Kukanov–Stoikov, R²≈65%)
6. **Broker alpha**: 1 basis point on \$200M/week ≈ \$1.04M/year


In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

pd.set_option("display.width", 110)
pd.set_option("display.max_columns", 20)
np.random.seed(7)
print("pandas", pd.__version__, "| numpy", np.__version__)


pandas 3.0.3 | numpy 2.3.5


## Part 1 — The Order Book: bid, ask, spread, depth

An **order book** is just every resting **limit order** for one stock, sorted by price:
the **highest bid first**, the **lowest ask first**. The book below uses the lecture's slide prices.

- **Best bid** = highest price a buyer will pay.
- **Best ask** (best offer) = lowest price a seller will accept.
- **Spread** = best ask − best bid (the cost of "crossing" immediately).
- **Depth** = how many shares are queued at each price level.


In [2]:
# Level-2 order book from the lecture slides (price, size in shares)
bids = pd.DataFrame({"price": [49.90, 49.85, 49.80, 49.75, 49.70],
                     "size":  [  600,   900,  1200,   800,  1500]})
asks = pd.DataFrame({"price": [50.10, 50.15, 50.20, 50.25, 50.30],
                     "size":  [  100,   700,  1000,   600,  1300]})
# convention: bids high->low, asks low->high
bids = bids.sort_values("price", ascending=False).reset_index(drop=True)
asks = asks.sort_values("price", ascending=True ).reset_index(drop=True)

best_bid = bids.loc[0, "price"]; best_ask = asks.loc[0, "price"]
spread   = best_ask - best_bid
mid      = (best_bid + best_ask) / 2
print("ORDER BOOK (top 5 levels each side)")
print("        BIDS                 ASKS")
for i in range(5):
    print(f"  {bids.loc[i,'size']:>5} @ {bids.loc[i,'price']:.2f}      "
          f"{asks.loc[i,'size']:>5} @ {asks.loc[i,'price']:.2f}")
print(f"\nBest bid = {best_bid:.2f} | Best ask = {best_ask:.2f}")
print(f"Spread   = {best_ask:.2f} - {best_bid:.2f} = {spread:.2f}  ({spread/mid*1e4:.1f} bps of mid {mid:.3f})")


ORDER BOOK (top 5 levels each side)
        BIDS                 ASKS
    600 @ 49.90        100 @ 50.10
    900 @ 49.85        700 @ 50.15
   1200 @ 49.80       1000 @ 50.20
    800 @ 49.75        600 @ 50.25
   1500 @ 49.70       1300 @ 50.30

Best bid = 49.90 | Best ask = 50.10
Spread   = 50.10 - 49.90 = 0.20  (40.0 bps of mid 50.000)


### Depth chart — the 'shape' of the book
Lecture point: the queue size is **smallest at the inside** (orders near the touch fill fast and
leave) and grows a few ticks out. We plot bid depth (left, green) vs ask depth (right, red).

In [3]:
fig, ax = plt.subplots(figsize=(8,3.2))
ax.barh(bids["price"], -bids["size"], height=0.035, color="#16a34a", label="bid size")
ax.barh(asks["price"],  asks["size"], height=0.035, color="#dc2626", label="ask size")
ax.axhline(mid, color="#334155", ls="--", lw=1, label=f"mid {mid:.3f}")
ax.set_xlabel("shares queued  (bids left / asks right)"); ax.set_ylabel("price")
ax.set_title("Order-book depth — smallest at the inside, grows outward")
ax.legend(fontsize=8); plt.tight_layout(); plt.savefig("chart_1_orderbook_depth.png", dpi=110); plt.show()
print("saved chart_1_orderbook_depth.png")


saved chart_1_orderbook_depth.png


C:\Users\hsaeed1\AppData\Local\Temp\ipykernel_39452\1852053339.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(fontsize=8); plt.tight_layout(); plt.savefig("chart_1_orderbook_depth.png", dpi=110); plt.show()


## Part 2 — A market order *walks the book* (step-by-step fills)

A **market order** demands immediacy: it takes the best price available, then the next, then the
next, until it is filled. So a **buy 400** does **not** all fill at the best ask 50.10 — it eats
each ask level in turn. This is *slippage* / *market impact* in action.

We compute the fills **row by row** so every share's price is visible.

In [4]:
def walk_book(side, qty, book):
    """Fill a market order against the opposite side, level by level."""
    rows, remaining = [], qty
    for _, lvl in book.iterrows():
        if remaining <= 0: break
        take = min(remaining, lvl["size"])
        rows.append({"level_price": lvl["price"], "shares_taken": take,
                     "cost_here": take * lvl["price"], "remaining_after": remaining - take})
        remaining -= take
    fills = pd.DataFrame(rows)
    avg = fills["cost_here"].sum() / fills["shares_taken"].sum()
    return fills, avg

fills, avg_px = walk_book("buy", 400, asks)
fills["cum_shares"] = fills["shares_taken"].cumsum()
print("MARKET BUY 400 — walking the ask side:")
print(fills.to_string(index=False))
print(f"\nAverage fill price = total cost {fills['cost_here'].sum():.2f} / 400 shares = {avg_px:.4f}")
print(f"Slippage vs best ask {best_ask:.2f} = {avg_px-best_ask:.4f} per share "
      f"= ${ (avg_px-best_ask)*400 :.2f} on 400 shares")


MARKET BUY 400 — walking the ask side:
 level_price  shares_taken  cost_here  remaining_after  cum_shares
       50.10         100.0     5010.0            300.0       100.0
       50.15         300.0    15045.0              0.0       400.0

Average fill price = total cost 20055.00 / 400 shares = 50.1375
Slippage vs best ask 50.10 = 0.0375 per share = $15.00 on 400 shares


A **limit order** is the opposite trade-off: total control over price, none over time. A
**buy-limit @ 49.90** only trades if a seller meets 49.90 or lower — otherwise it just sits in the
book and *provides* liquidity (you'd earn the spread if a market sell hits you).

## Part 3 — VWAP (Volume-Weighted Average Price) — the exact Excel numbers

VWAP is the desk's benchmark: *did I buy below the day's volume-weighted price (good) or above it
(bad)?* It weights each trade price by its size. These are the **exact four trades from the
lecture's Excel sheet** — the answer is **14.205**.

In [5]:
trades = pd.DataFrame({"price":  [14.00, 14.50, 14.20, 14.35],
                       "volume": [ 1100,   600,  1800,   400]})
trades["price_x_volume"] = trades["price"] * trades["volume"]
vwap = trades["price_x_volume"].sum() / trades["volume"].sum()
print("VWAP step table (price x volume, then divide by total volume):")
print(trades.to_string(index=False))
print(f"\nSum(price x volume) = {trades['price_x_volume'].sum():.0f}")
print(f"Sum(volume)         = {trades['volume'].sum():.0f}")
print(f"VWAP = {trades['price_x_volume'].sum():.0f} / {trades['volume'].sum():.0f} = {vwap:.3f}")


VWAP step table (price x volume, then divide by total volume):
 price  volume  price_x_volume
 14.00    1100         15400.0
 14.50     600          8700.0
 14.20    1800         25560.0
 14.35     400          5740.0

Sum(price x volume) = 55400
Sum(volume)         = 3900
VWAP = 55400 / 3900 = 14.205


### Moving VWAP on a simulated session
A real desk tracks a **running** VWAP through the day. We simulate one session of (price, volume)
ticks and overlay price, the cumulative VWAP, and a rolling 20-tick MVWAP.

In [6]:
n = 200
t = np.arange(n)
px = 14.20 + np.cumsum(np.random.normal(0, 0.012, n))          # simulated price path
vol = np.random.randint(200, 2000, n)                          # simulated tick volume
df = pd.DataFrame({"price": px, "volume": vol})
df["cum_vwap"] = (df.price*df.volume).cumsum() / df.volume.cumsum()
df["mvwap20"]  = (df.price*df.volume).rolling(20).sum() / df.volume.rolling(20).sum()

fig, ax = plt.subplots(figsize=(8,3.2))
ax.plot(t, df.price,    color="#94a3b8", lw=1,   label="price (sim)")
ax.plot(t, df.cum_vwap, color="#2563eb", lw=1.8, label="cumulative VWAP")
ax.plot(t, df.mvwap20,  color="#f59e0b", lw=1.5, label="20-tick MVWAP")
ax.set_title("VWAP vs MVWAP over a simulated session"); ax.set_xlabel("tick"); ax.set_ylabel("price")
ax.legend(fontsize=8); plt.tight_layout(); plt.savefig("chart_2_vwap.png", dpi=110); plt.show()
print("Buying BELOW the blue VWAP line = a good fill; above = a bad fill.")
print("saved chart_2_vwap.png")


Buying BELOW the blue VWAP line = a good fill; above = a bad fill.
saved chart_2_vwap.png


C:\Users\hsaeed1\AppData\Local\Temp\ipykernel_39452\2037729039.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(fontsize=8); plt.tight_layout(); plt.savefig("chart_2_vwap.png", dpi=110); plt.show()


## Part 4 — Execution time-horizon rule (TTH / PTH)

Big orders can't be dumped as one market order (huge impact). The desk slices them over time. The
lecture's rule decides whether there's *enough time* to finish:

- **Theoretical Time Horizon** TTH = Order Shares ÷ (Vol Ratio × Min Vol)
- Vol Ratio = 15% (High urgency) / 7.5% (Medium) / 2.5% (Low)
- **Practical Time Horizon** PTH = min(time left, TTH), clamped to [60, 120] min
- If TTH/PTH > 2 → not enough time, bounce the order back.

Lecture example: 10,000 shares, High urgency, MinVol = 500 shares/min → **TTH = 134 min → PTH = 120**.

In [7]:
def horizon(order_shares, urgency, min_vol_per_min, time_left=120, lo=60, hi=120):
    vr = {"High":0.15, "Medium":0.075, "Low":0.025}[urgency]
    tth = order_shares / (vr * min_vol_per_min)
    pth = min(time_left, tth); pth = max(lo, min(hi, pth))
    return vr, tth, pth

steps = [("order shares", "10000"),
         ("urgency -> Vol Ratio", "High -> 0.15"),
         ("Min Vol (shares/min, from 21-day avg)", "500"),
         ("TTH = 10000 / (0.15 x 500)", f"{10000/(0.15*500):.0f} min"),
         ("PTH = min(time_left 120, TTH 134), clamp[60,120]", "120 min"),
         ("TTH/PTH = 134/120", f"{(10000/(0.15*500))/120:.2f}"),
         ("verdict (>2 = bounce)", "1.12  -> OK, accept")]
print("EXECUTION TIME-HORIZON — step ladder")
for k,v in steps: print(f"  {k:<48} {v}")
vr,tth,pth = horizon(10000,"High",500)
print(f"\nFunction check: TTH={tth:.0f} min, PTH={pth:.0f} min")


EXECUTION TIME-HORIZON — step ladder
  order shares                                     10000
  urgency -> Vol Ratio                             High -> 0.15
  Min Vol (shares/min, from 21-day avg)            500
  TTH = 10000 / (0.15 x 500)                       133 min
  PTH = min(time_left 120, TTH 134), clamp[60,120] 120 min
  TTH/PTH = 134/120                                1.11
  verdict (>2 = bounce)                            1.12  -> OK, accept

Function check: TTH=133 min, PTH=120 min


## Part 5 — Order-Flow Imbalance → price impact (the exercise paper)

The in-class reading (Cont, Kukanov & Stoikov 2011) shows mid-price changes are driven not by
*volume* but by **order-flow imbalance (OFI)** — net change in the bid/ask queues — and the relation
is **linear**, with R²≈65% on real data. We reproduce that shape on a simulated tick stream:
OFI up (more buying pressure) → mid-price up.

In [8]:
m = 600
# simulate per-tick order-flow imbalance and a linear price response + noise
ofi = np.random.normal(0, 1.0, m)               # net (bid adds - ask adds), standardised
impact_coef = 0.8                                # slope ~ 1/depth
dmid = impact_coef*ofi + np.random.normal(0, 0.6, m)   # mid-price change (ticks)

b1, b0 = np.polyfit(ofi, dmid, 1)
pred = b1*ofi + b0
ss_res = np.sum((dmid-pred)**2); ss_tot = np.sum((dmid-dmid.mean())**2)
r2 = 1 - ss_res/ss_tot
print(f"Fitted   Δmid = {b1:.3f} * OFI + {b0:.3f}")
print(f"Slope (price-impact coeff) = {b1:.3f}  (inversely proportional to depth)")
print(f"R^2 = {r2:.2f}   (paper reports ~0.65 on real TAQ data)")

fig, ax = plt.subplots(figsize=(7.6,3.3))
ax.scatter(ofi, dmid, s=8, alpha=0.4, color="#2563eb", label="ticks")
xs = np.linspace(ofi.min(), ofi.max(), 50)
ax.plot(xs, b1*xs+b0, color="#dc2626", lw=2, label=f"fit  slope={b1:.2f}, R²={r2:.2f}")
ax.axhline(0, color="#cbd5e1", lw=.8); ax.axvline(0, color="#cbd5e1", lw=.8)
ax.set_xlabel("order-flow imbalance (OFI)"); ax.set_ylabel("mid-price change (ticks)")
ax.set_title("Linear price impact of order-flow imbalance"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("chart_3_ofi_impact.png", dpi=110); plt.show()
print("saved chart_3_ofi_impact.png")


Fitted   Δmid = 0.769 * OFI + 0.027
Slope (price-impact coeff) = 0.769  (inversely proportional to depth)
R^2 = 0.62   (paper reports ~0.65 on real TAQ data)


saved chart_3_ofi_impact.png


C:\Users\hsaeed1\AppData\Local\Temp\ipykernel_39452\3507715740.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("chart_3_ofi_impact.png", dpi=110); plt.show()


## Part 6 — Broker alpha: why 1 basis point matters

The Singapore-hedge-fund example: trading **\$200M/week**, a broker whose tech saves **1 basis
point** (1 cent per \$100) per trade adds pure *broker alpha*. Over a year that is real money.

In [9]:
weekly_notional = 200e6      # $200M traded per week
bps = 1                       # 1 basis point saved per trade
weeks = 52
saving_per_week = weekly_notional * (bps/1e4)
annual = saving_per_week * weeks
print("BROKER ALPHA — step ladder")
print(f"  weekly notional traded        ${weekly_notional:,.0f}")
print(f"  x 1 basis point (1/10000)     {bps/1e4}")
print(f"  = saving per week             ${saving_per_week:,.0f}")
print(f"  x 52 weeks                    = ${annual:,.0f} / year")
print("\nSame strategy alpha, just a faster broker -> ~$1.04M extra per year.")


BROKER ALPHA — step ladder
  weekly notional traded        $200,000,000
  x 1 basis point (1/10000)     0.0001
  = saving per week             $20,000
  x 52 weeks                    = $1,040,000 / year

Same strategy alpha, just a faster broker -> ~$1.04M extra per year.


## Summary — what to remember
- **Order book** = resting limit orders, **highest bid / lowest ask first**; **spread** = ask−bid;
  depth is thin at the inside.
- **Market order** = certain fill, uncertain price (walks the book → slippage). **Limit order** =
  certain price, uncertain fill (provides liquidity).
- **Price-time priority**: best price first, then earliest order.
- **VWAP** is the execution benchmark (buy below it = good). **Time-horizon** rule slices big orders.
- Price moves with **order-flow imbalance** (linear), not raw volume.
- Tiny edges (**1 bp** of broker alpha) compound to millions at scale — *every cent matters*.
